# SALM Fine-tuning 04 · What does the user want? Smart-home intents (SLURP)

> **Task** spoken request → one of 60 intents · **Data** SLURP (40 h, headset + far-field)
> **Metric** accuracy ↑ · **Result** 50.57% → **87.91%** (+73.9%) · **Time** ~2 h

Part of the [SALM fine-tuning series](README.md); see [00_Overview_and_Setup](00_Overview_and_Setup.ipynb) first.

## The problem: an assistant that invents its own menu

"Olly, wake me up at seven." "What's the weather tomorrow?" "Turn the lights down." SLURP has 60 intents like
`alarm_set`, `weather_query` and `iot_hue_lightdim`. We put the full list in the prompt and ask for one label.

The base model understands the requests, but it **invents labels that aren't on the menu**: `cooking_query` 223 times
in the test decode, `general_query`, `movie_query`, `remindme`… Plausible, readable, and scored as wrong. The goal is
a model that always picks a label from the menu, and picks the right one.

> Memory note: this recipe uses 96k-token batches (~160 GB during training). Run it with the GPU to itself.

In [ ]:
import json, os, shlex, subprocess, sys, time
from pathlib import Path


def find_nemo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "examples" / "speechlm2" / "finetuning").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate the NeMo source tree from {start}. Open this notebook from inside the repo checkout.")


NEMO_ROOT = find_nemo_root(Path.cwd())
FT = NEMO_ROOT / "examples" / "speechlm2" / "finetuning"              # the tools used below
WORK = Path(os.environ.get("SALM_FT_WORK", NEMO_ROOT / "tutorials" / "speechlm2" / "finetuning" / "workdir")).absolute()
os.environ["SALM_FT_WORK"] = str(WORK)                                # data, manifests, experiments, exports
os.environ.pop("MPLBACKEND", None)  # the kernel's inline plotting backend would break matplotlib in child processes
if not os.environ.get("SALM_CHECKPOINT"):
    raise RuntimeError(
        "SALM_CHECKPOINT is not set. Point it at your SALM checkpoint: a directory in HuggingFace format "
        "exported by SALMAutomodel. Run `export SALM_CHECKPOINT=/path/to/checkpoint` before "
        "starting Jupyter, or set os.environ['SALM_CHECKPOINT'] in a cell above this one."
    )
CKPT = Path(os.environ["SALM_CHECKPOINT"]).expanduser().absolute()  # base SALM checkpoint (HuggingFace format)
if not CKPT.is_dir():
    raise RuntimeError(f"SALM_CHECKPOINT={CKPT} is not a directory. It must be a SALM checkpoint in HuggingFace format.")
PYTHON = os.environ.get("TRAIN_PYTHON", "python")                     # NeMo training environment
VLLM_PYTHON = os.environ.get("VLLM_PYTHON", "python")                 # vLLM 0.28 environment (ideally a separate one)
M, EXP, EXPORT, EVAL = (WORK / d for d in ("manifests", "experiments", "exports", "eval"))
q = shlex.quote                                                        # shell-quote prompts (keeps non-ASCII intact)
for d in (M, EXP, EXPORT, EVAL, WORK / "logs", WORK / "configs"):
    d.mkdir(parents=True, exist_ok=True)


RETRIES = int(os.environ.get("SALM_FT_RETRIES", 3))                   # raise when other jobs share the GPU


def sh(cmd, log=None, retries=1):
    """Run a shell command from the NeMo root. With `log`, output goes to a file and only its tail is shown."""
    for attempt in range(1, retries + 1):
        print(f"$ {cmd}" + (f"   > {log}" if log else ""), flush=True)
        if log:
            with open(WORK / "logs" / log, "w") as f:
                rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT, stdout=f, stderr=subprocess.STDOUT).returncode
            tail = (WORK / "logs" / log).read_text(errors="ignore").splitlines()
            print("\n".join(l for l in tail[-400:] if l.startswith(("[eval]", "[export]", "[scores]", "[merge]", "wrote")))[-3000:])
        else:
            rc = subprocess.run(cmd, shell=True, cwd=NEMO_ROOT).returncode
        if rc == 0:
            return
        if attempt < retries:  # e.g. a vLLM engine starting while another job allocates GPU memory
            print(f"(attempt {attempt} failed, retrying in 90 s)")
            time.sleep(90)
    raise RuntimeError(f"command failed (rc={rc}): {cmd}" + (f"; see {WORK / 'logs' / log}" if log else ""))


def evaluate(model, manifest, name, task_args, extra=""):
    """Decode `manifest` with vLLM (greedy) and return the metric summary.

    Re-running a cell reuses a previous result only if its fingerprint (model, manifest, prompt, decoder and scoring
    settings, code revision) matches; a result produced from different inputs is an error, never silently reused.
    """
    out = EVAL / f"{name}.jsonl"
    sh(f"{VLLM_PYTHON} {FT}/vllm_task_eval.py --model {model} --manifest {manifest} --out {out} {task_args} {extra} "
       f"--reuse", log=f"{name}.log", retries=RETRIES)
    return json.load(open(f"{out}.summary.json"))


def tidy(keep, name):
    """Free disk: delete this tutorial's other merged exports (~63 GB each) and leftover training checkpoints."""
    import shutil

    for p in EXPORT.glob(f"{name}_*"):
        if p != keep:
            shutil.rmtree(p)
    shutil.rmtree(EXP / name / "checkpoints", ignore_errors=True)
    print(f"kept {keep}; removed the other {name} exports and checkpoints")


def samples(name, n=3, keys=("text", "pred_text")):
    """Print a few decoded examples."""
    rows = [json.loads(l) for l in open(EVAL / f"{name}.jsonl")]
    for r in rows[:n]:
        print("  |  ".join(f"{k}: {str(r.get(k))[:110]}" for k in keys))


def result(name, value, expected, tol):
    """Compare a number with the expected result, and record it in results.json."""
    ok = abs(value - expected) <= tol
    print(f"{'✅' if ok else '❌'} {name}: {value:.4f}  (expected: {expected} ± {tol})")
    res = WORK / "results.json"
    d = json.loads(res.read_text()) if res.exists() else {}
    d[name] = {"value": value, "expected": expected, "tolerance": tol, "ok": ok}
    res.write_text(json.dumps(d, indent=1))
    return ok


print("NeMo root:", NEMO_ROOT, "\nwork dir: ", WORK, "\ncheckpoint:", CKPT)

## 1. Data, and a label bug in the dataset itself

The obvious `intent` column is **not** the label. For ~1.6k utterances it holds only the *action* (`query` for weather,
calendar and news queries alike; bare `set` and `remove`), which gives 91 inconsistent labels, some of which appear in
test but never in train. SLURP's official evaluation scores `scenario_action`, and that gives exactly **60** clean
labels. The prep script downloads the official annotations and rebuilds labels that way.

In [ ]:
d = M / "slurp"
sh(f"{PYTHON} {FT}/prepare_benchmarks.py slurp", log="prep_slurp.log")
sh(f"{PYTHON} {FT}/prepare_benchmarks.py librispeech", log="prep_ls.log")
import collections
labels = open(d / "labels.txt").read().split()
test_labels = collections.Counter(json.loads(l)["text"] for l in open(d / "test.json"))
print(len(labels), "intents, e.g.", labels[:6], "| test labels missing from train:", set(test_labels) - set(labels))

> Before you trust a label column, count its values and compare them with the paper.

## 2. Baseline

Answers are mapped leniently onto the label set (underscore = space, synonyms, a unique action suffix like `maths` →
`qa_maths`). Anything else counts as wrong and is reported as "invalid".

In [ ]:
TASK = f"--task cls --max-tokens 16 --labels {d}/labels.txt"
P2 = "{context} Labels are scenario_action. Reply with the label only."
base = {}
for p, extra in (("p1", ""), ("p2", f"--prompt-template {q(P2)}")):
    for s, man in (("dev", "devel"), ("test", "test")):
        base[p, s] = evaluate(CKPT, d / f"{man}.json", f"slu_base_{p}_{s}", TASK, extra)
    print(f"{p}  dev {100 * base[p, 'dev']['score']:5.2f}%   test {100 * base[p, 'test']['score']:5.2f}%   "
          f"invalid {base[p, 'test']['invalid_rate']:.1%}")
BEST_P = max(("p1", "p2"), key=lambda p: base[p, "dev"]["score"])  # prompt chosen on dev
BASELINE = 100 * base[BEST_P, "test"]["score"]  # its test score: the baseline fine-tuning is compared with
ORACLE = 100 * max(base[p, "test"]["score"] for p in ("p1", "p2"))  # best prompt chosen on test: an upper bound
print(f"baseline prompt (chosen on dev): {BEST_P}; test-oracle (best prompt on test): {ORACLE:.2f}")
print(f"\nbaseline = {BASELINE:.2f}%")
pairs = " ".join(f"{d}/{s}.json {d}/{s}_ft.json" for s in ("train", "devel", "test"))
sh(f"{PYTHON} {FT}/set_context.py --template {q(P2)} {pairs}")

## 3. Config: the menu is expensive

The 60-label list is ~450 tokens, about 10× the audio, so an epoch is ~25M tokens. Big batches (96k tokens), two
epochs.

In [ ]:
CFG = WORK / "configs" / "slu.yaml"
sh(f"{PYTHON} {FT}/make_ft_config.py --checkpoint {CKPT} --prompt manifest --out {CFG} --exp-dir {EXP / 'slu'} "
   f"--train-manifest {d}/train_ft.json --val-manifest {d}/devel_ft.json --val-name slu_dev "
   f"--train-encoder --train-proj --lr 1e-4 --warmup-steps 50 --max-steps 500 "
   f"--batch-tokens 96000 --limit-train-batches 125 --limit-val-batches 25 --save-top-k -1")

## 4. Train (~5.3 s/step, ~50 min)

In [ ]:
sh(f"{PYTHON} {FT}/run_training.py --config {CFG} --preflight --smoke --keep-steps 250 375 500")

## 5. Select on dev

Expected dev accuracy: step 250 → 84.97%, step 375 (lowest `val_loss`, 0.222) → 86.63%, **step 500 → 87.19%** (the
highest `val_loss` of the three, 0.246). The lowest `val_loss` is not the best model.

In [ ]:
dev = {}
for step in (250, 375, 500):
    out = EXPORT / f"slu_s{step}"
    sh(f"{PYTHON} {FT}/export_checkpoint.py --exp-dir {EXP / 'slu'} --step {step} --base-checkpoint {CKPT} "
       f"--out {out} --delete-ckpt", log=f"export_slu_{step}.log")
    dev[step] = 100 * evaluate(out, d / "devel_ft.json", f"slu_s{step}_dev", TASK)["score"]
    print(f"step {step}: dev accuracy {dev[step]:.2f}%")
BEST = max(dev, key=dev.get)

## 6. Test, once

In [ ]:
MODEL = EXPORT / f"slu_s{BEST}"
ft = evaluate(MODEL, d / "test_ft.json", "slu_TEST", TASK)
guard = evaluate(MODEL, M / "librispeech" / "test_clean.json", "slu_librispeech", "--task asr --normalizer english")
FT_ACC = 100 * ft["score"]
print(f"test accuracy: base {BASELINE:.2f}% → fine-tuned {FT_ACC:.2f}% ({(FT_ACC - BASELINE) / BASELINE:+.1%}); "
      f"invalid answers {ft['invalid_rate']:.2%}; LibriSpeech WER {100 * guard['score']:.2f}%")
samples("slu_TEST", n=4, keys=("transcript", "text", "pred_text"))
result("04 baseline test accuracy", BASELINE, 50.57, 0.5)
result("04 fine-tuned test accuracy", FT_ACC, 87.91, 1.0)

Expected results:

| | test accuracy | per-intent mean | invalid answers | LibriSpeech WER |
|---|---:|---:|---:|---:|
| base, prompt chosen on dev (p2) | 50.57% | 45.35% | 4.8% | 1.54% |
| **fine-tuned** | **87.91%** | 85.58% | 0.02% | 1.78% |

**+73.9% relative.** The prompt chosen on dev (p2) is also the test-oracle. The fine-tuned model stops inventing
labels.

## Tidy up

Keep the selected model; delete the other exports (~63 GB each) and any leftover training checkpoints.

In [ ]:
tidy(keep=MODEL, name="slu")

## Takeaways

1. **Audit labels against the official protocol.** A dataset's own field can be wrong.
2. **Long prompts cost compute.** Size `batch_tokens` on the total, not on audio hours.
3. **A closed label set is learned fast.** Invalid answers went from 4.8% to 0.02%.